# Phase 3C.1 — Evaluation Protocol Freeze

**Experiment:** `phase_3c1_evaluation_protocol_freeze`

This follow-up freezes the leakage-controlled evaluation protocol after resolving the Phase 3C split-representation discrepancy. It fixes artifact-path resolution, defines one canonical feature-group encoding, repeats the primary split to verify deterministic membership, and retains Protocol A/C for context.

### Decision set
- **A — Random row split:** historical Phase 2 baseline; retained for continuity only.
- **B — Deterministic feature-group-aware split, all groups:** candidate and, if repeatability checks pass, official primary benchmark.
- **C — Feature-group-aware split with conflicting groups excluded:** sensitivity analysis only.
- **D — Derived-label policy:** not admissible without external label authority.

No production code, canonical labels, or DVC-tracked dataset content is modified by this experiment.


## Phase 3C.1 objectives

1. Resolve the earlier `GroupShuffleSplit` discrepancy by recognizing that seeded sampling depends on the ordering of group identifiers, even when the underlying feature groups are mathematically identical.
2. Define **one canonical group encoding**: lexicographically sorted complete 30-feature tuples mapped to integer IDs.
3. Run the canonical Protocol B split twice with the same dataset, group IDs, `test_size`, and `random_state`.
4. Require identical train/test membership fingerprints and identical metrics across the two runs before freezing Protocol B.
5. Use a repository-root-aware artifact path so execution from the repository root, `notebooks/`, or another descendant directory does not create a nested `notebooks/notebooks/...` path.


In [1]:
from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "Result"
EXPECTED_SHA256 = "a4b16abbd8610e4f53fd63e8eb3da793157a961111ed5e9d9d8afa21af866995"


def find_repo_root(start=None):
    """Find the project root without depending on the notebook working directory."""
    start_path = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "docs").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start the notebook from within the project tree "
        "containing pyproject.toml and docs/."
    )


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "raw" / "phisingData.csv"
ARTIFACT_PATH = REPO_ROOT / "notebooks" / "evaluation" / "phase_3c_evaluation_protocol_comparison_freezed.json"
NOTEBOOK_PATH = REPO_ROOT / "notebooks" / "17_evaluation_protocol_freeze.ipynb"


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def dataframe_fingerprint(frame):
    payload = pd.util.hash_pandas_object(frame, index=True).values.tobytes()
    return hashlib.sha256(payload).hexdigest()


def membership_fingerprint(indices):
    arr = np.sort(np.asarray(indices, dtype=np.int64))
    return hashlib.sha256(arr.tobytes()).hexdigest()


if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Canonical phisingData.csv was not found at {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
actual_sha256 = sha256_file(DATA_PATH)

print("Repository root:", REPO_ROOT)
print("Dataset:", DATA_PATH)
print("Artifact:", ARTIFACT_PATH)
print("Shape:", df.shape)
print("SHA256:", actual_sha256)
print("Target counts:")
print(df[TARGET].value_counts().sort_index())

if actual_sha256.lower() != EXPECTED_SHA256.lower():
    raise ValueError(f"Dataset SHA256 mismatch. Expected {EXPECTED_SHA256}, got {actual_sha256}.")

feature_cols = [c for c in df.columns if c != TARGET]
if len(feature_cols) != 30:
    raise ValueError(f"Expected 30 feature columns, found {len(feature_cols)}.")


Repository root: E:\Projects\Network security log triage agent
Dataset: E:\Projects\Network security log triage agent\data\raw\phisingData.csv
Artifact: E:\Projects\Network security log triage agent\notebooks\evaluation\phase_3c_evaluation_protocol_comparison_freezed.json
Shape: (11055, 31)
SHA256: a4b16abbd8610e4f53fd63e8eb3da793157a961111ed5e9d9d8afa21af866995
Target counts:
Result
-1    4898
 1    6157
Name: count, dtype: int64


In [2]:

dataset_metadata = {
    "path": str(DATA_PATH),
    "sha256": actual_sha256,
    "shape": list(df.shape),
    "dataframe_fingerprint": dataframe_fingerprint(df),
    "target": TARGET,
    "feature_count": len(feature_cols),
    "exact_duplicate_rows": int(df.duplicated(keep=False).sum()),
    "unique_full_rows": int(df.drop_duplicates().shape[0]),
}
dataset_metadata


{'path': 'E:\\Projects\\Network security log triage agent\\data\\raw\\phisingData.csv',
 'sha256': 'a4b16abbd8610e4f53fd63e8eb3da793157a961111ed5e9d9d8afa21af866995',
 'shape': [11055, 31],
 'dataframe_fingerprint': '885e849647c229a5a08975c61e39a2e3d94b08ac740a6e975319dbff12703b72',
 'target': 'Result',
 'feature_count': 30,
 'exact_duplicate_rows': 7843,
 'unique_full_rows': 5849}

## 1. Feature-group construction

A feature group is defined by the complete 30-feature vector, excluding `Result`.

For reconciliation we retain the earlier legacy and hash representations, but **Protocol B uses only the canonical representation below**. The canonical representation is:

1. convert each complete feature vector to a tuple;
2. sort the unique tuples lexicographically;
3. map each tuple to an integer ID according to that sorted order.

The important reproducibility invariant is therefore not that unrelated identifier encodings produce the same seeded split. It is that the **same canonical encoding produces the same split every time**.


In [3]:
feature_tuples = df[feature_cols].apply(tuple, axis=1)

# Legacy first-seen IDs, retained only to explain historical split differences.
legacy_groups = pd.Series(
    pd.factorize(feature_tuples, sort=False)[0],
    index=df.index,
    name="legacy_group",
)

# Canonical deterministic IDs used by Protocol B.
unique_feature_tuples = sorted(set(feature_tuples.tolist()))
canonical_group_map = {t: i for i, t in enumerate(unique_feature_tuples)}
canonical_groups = feature_tuples.map(canonical_group_map).astype("int64")
canonical_groups.name = "canonical_sorted_group"

# Stable content hashes, retained as an independent representation check.
def tuple_digest(values):
    return hashlib.sha256(repr(tuple(values)).encode("utf-8")).hexdigest()

hash_groups = feature_tuples.map(tuple_digest)
hash_groups.name = "hash_group"

print("Total feature groups:", canonical_groups.nunique())
print("Legacy group count:", legacy_groups.nunique())
print("Canonical group count:", canonical_groups.nunique())
print("Stable-hash group count:", hash_groups.nunique())


Total feature groups: 5785
Legacy group count: 5785
Canonical group count: 5785
Stable-hash group count: 5785


In [4]:
def split_indices(groups):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    train_idx, test_idx = next(
        splitter.split(df[feature_cols], df[TARGET], groups=groups)
    )
    return np.asarray(train_idx), np.asarray(test_idx)


def split_fingerprints(train_idx, test_idx):
    return {
        "train": membership_fingerprint(train_idx),
        "test": membership_fingerprint(test_idx),
    }


def index_overlap(a, b):
    return int(len(set(a).intersection(set(b))))

legacy_train_idx, legacy_test_idx = split_indices(legacy_groups)
canonical_train_idx_1, canonical_test_idx_1 = split_indices(canonical_groups)
canonical_train_idx_2, canonical_test_idx_2 = split_indices(canonical_groups)
hash_train_idx, hash_test_idx = split_indices(hash_groups)

reconciliation = {
    "legacy_vs_canonical": {
        "train_rows_equal": bool(np.array_equal(np.sort(legacy_train_idx), np.sort(canonical_train_idx_1))),
        "test_rows_equal": bool(np.array_equal(np.sort(legacy_test_idx), np.sort(canonical_test_idx_1))),
        "train_index_overlap": index_overlap(legacy_train_idx, canonical_train_idx_1),
        "test_index_overlap": index_overlap(legacy_test_idx, canonical_test_idx_1),
    },
    "canonical_repeatability": {
        "train_rows_equal": bool(np.array_equal(np.sort(canonical_train_idx_1), np.sort(canonical_train_idx_2))),
        "test_rows_equal": bool(np.array_equal(np.sort(canonical_test_idx_1), np.sort(canonical_test_idx_2))),
    },
    "canonical_vs_hash": {
        "train_rows_equal": bool(np.array_equal(np.sort(canonical_train_idx_1), np.sort(hash_train_idx))),
        "test_rows_equal": bool(np.array_equal(np.sort(canonical_test_idx_1), np.sort(hash_test_idx))),
        "train_index_overlap": index_overlap(canonical_train_idx_1, hash_train_idx),
        "test_index_overlap": index_overlap(canonical_test_idx_1, hash_test_idx),
    },
}
reconciliation


{'legacy_vs_canonical': {'train_rows_equal': False,
  'test_rows_equal': False,
  'train_index_overlap': 6965,
  'test_index_overlap': 446},
 'canonical_repeatability': {'train_rows_equal': True,
  'test_rows_equal': True},
 'canonical_vs_hash': {'train_rows_equal': False,
  'test_rows_equal': False,
  'train_index_overlap': 7029,
  'test_index_overlap': 515}}

In [5]:
def shared_group_count(groups, train_idx, test_idx):
    train_groups = set(pd.Series(groups).iloc[train_idx])
    test_groups = set(pd.Series(groups).iloc[test_idx])
    return int(len(train_groups.intersection(test_groups)))


def evaluate_indices(frame, groups, train_idx, test_idx, protocol_name):
    feature_columns = [c for c in frame.columns if c != TARGET]
    X = frame[feature_columns].copy()
    y = frame[TARGET].map({-1: 0, 1: 1}).astype(int)

    imputer = KNNImputer(n_neighbors=3, weights="uniform")
    X_train = imputer.fit_transform(X.iloc[train_idx])
    X_test = imputer.transform(X.iloc[test_idx])

    model = RandomForestClassifier(
        n_estimators=128,
        criterion="gini",
        bootstrap=True,
        max_depth=None,
        max_features="sqrt",
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y.iloc[train_idx])
    predictions = model.predict(X_test)

    return {
        "protocol": protocol_name,
        "train_rows": int(len(train_idx)),
        "test_rows": int(len(test_idx)),
        "train_target_counts": {str(k): int(v) for k, v in y.iloc[train_idx].value_counts().sort_index().items()},
        "test_target_counts": {str(k): int(v) for k, v in y.iloc[test_idx].value_counts().sort_index().items()},
        "shared_feature_groups": shared_group_count(groups, train_idx, test_idx),
        "accuracy": float(accuracy_score(y.iloc[test_idx], predictions)),
        "f1": float(f1_score(y.iloc[test_idx], predictions)),
        "precision": float(precision_score(y.iloc[test_idx], predictions)),
        "recall": float(recall_score(y.iloc[test_idx], predictions)),
        "confusion_matrix": confusion_matrix(y.iloc[test_idx], predictions).tolist(),
        "train_membership_fingerprint": membership_fingerprint(train_idx),
        "test_membership_fingerprint": membership_fingerprint(test_idx),
    }


### Reconciliation interpretation

The legacy, canonical, and hash representations encode the same mathematical feature groups, but `GroupShuffleSplit` samples **group positions**. Therefore, different group-ID orderings can legitimately produce different seeded partitions.

For Phase 3C.1, the reproducibility requirement is instead:

- canonical group IDs are generated by one documented algorithm;
- the same canonical dataset and configuration produce identical train/test membership on repeated execution;
- train/test contain zero shared canonical feature groups.

Agreement between unrelated group-ID encodings is **not** required for protocol freeze.


In [6]:
tuple_series = df[feature_cols].apply(tuple, axis=1)

def group_relation_is_valid(groups):
    checks = tuple_series.groupby(groups).nunique(dropna=False)
    return bool(checks.max() == 1)

group_equivalence_check = {
    "legacy_valid": group_relation_is_valid(legacy_groups),
    "canonical_valid": group_relation_is_valid(canonical_groups),
    "hash_valid": group_relation_is_valid(hash_groups),
    "legacy_group_count": int(legacy_groups.nunique()),
    "canonical_group_count": int(canonical_groups.nunique()),
    "hash_group_count": int(hash_groups.nunique()),
    "canonical_group_ids_are_contiguous": bool(
        np.array_equal(
            np.sort(canonical_groups.unique()),
            np.arange(canonical_groups.nunique()),
        )
    ),
}
group_equivalence_check


{'legacy_valid': True,
 'canonical_valid': True,
 'hash_valid': True,
 'legacy_group_count': 5785,
 'canonical_group_count': 5785,
 'hash_group_count': 5785,
 'canonical_group_ids_are_contiguous': True}


## 2. Protocol A — random row split

This reproduces the Phase 2 seeded evaluation. It is retained as the historical baseline and makes duplicate overlap measurable.

It is **not leakage-controlled** when duplicate feature groups span train and test.


In [7]:
train_idx_a, test_idx_a = train_test_split(
    np.arange(len(df)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=None,
)

protocol_a = evaluate_indices(
    df, canonical_groups, train_idx_a, test_idx_a,
    "A_random_row_split",
)

train_groups_a = set(canonical_groups.iloc[train_idx_a])
test_groups_a = set(canonical_groups.iloc[test_idx_a])
test_rows_with_train_duplicate = int(canonical_groups.iloc[test_idx_a].isin(train_groups_a).sum())

protocol_a["shared_feature_groups"] = int(len(train_groups_a.intersection(test_groups_a)))
protocol_a["test_rows_with_train_duplicate"] = test_rows_with_train_duplicate
protocol_a["test_duplicate_overlap_rate"] = float(test_rows_with_train_duplicate / len(test_idx_a))
protocol_a


{'protocol': 'A_random_row_split',
 'train_rows': 8844,
 'test_rows': 2211,
 'train_target_counts': {'0': 3942, '1': 4902},
 'test_target_counts': {'0': 956, '1': 1255},
 'shared_feature_groups': 1143,
 'accuracy': 0.968340117593849,
 'f1': 0.9723320158102767,
 'precision': 0.9647058823529412,
 'recall': 0.9800796812749004,
 'confusion_matrix': [[911, 45], [25, 1230]],
 'train_membership_fingerprint': '4f86d11e876c2ef26f0888c1ce690698758915cc3acb0640b0ec4f71631e81fb',
 'test_membership_fingerprint': '5ee41c88306eb5585eae19ba3d1a7af603f44fc13f3abef15e3cb039e9ad60da',
 'test_rows_with_train_duplicate': 1447,
 'test_duplicate_overlap_rate': 0.6544549977385798}

## 3. Protocol B — canonical deterministic feature-group-aware split

This is the candidate **official primary evaluation protocol**.

Identical feature vectors cannot cross the train/test boundary. Canonical group IDs are assigned from lexicographically sorted complete feature tuples. The split is then repeated with exactly the same canonical groups and configuration.

The freeze criterion is **repeatable membership + zero shared feature groups**, not agreement with an alternate group-ID encoding.


In [8]:
def shared_group_count(groups, train_idx, test_idx):
    train_groups = set(pd.Series(groups).iloc[train_idx])
    test_groups = set(pd.Series(groups).iloc[test_idx])
    return int(len(train_groups.intersection(test_groups)))


def evaluate_indices(frame, groups, train_idx, test_idx, protocol_name):
    feature_columns = [c for c in frame.columns if c != TARGET]
    X = frame[feature_columns].copy()
    y = frame[TARGET].map({-1: 0, 1: 1}).astype(int)

    imputer = KNNImputer(n_neighbors=3, weights="uniform")
    X_train = imputer.fit_transform(X.iloc[train_idx])
    X_test = imputer.transform(X.iloc[test_idx])

    model = RandomForestClassifier(
        n_estimators=128,
        criterion="gini",
        bootstrap=True,
        max_depth=None,
        max_features="sqrt",
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y.iloc[train_idx])
    predictions = model.predict(X_test)

    return {
        "protocol": protocol_name,
        "train_rows": int(len(train_idx)),
        "test_rows": int(len(test_idx)),
        "train_target_counts": {str(k): int(v) for k, v in y.iloc[train_idx].value_counts().sort_index().items()},
        "test_target_counts": {str(k): int(v) for k, v in y.iloc[test_idx].value_counts().sort_index().items()},
        "shared_feature_groups": shared_group_count(groups, train_idx, test_idx),
        "accuracy": float(accuracy_score(y.iloc[test_idx], predictions)),
        "f1": float(f1_score(y.iloc[test_idx], predictions)),
        "precision": float(precision_score(y.iloc[test_idx], predictions)),
        "recall": float(recall_score(y.iloc[test_idx], predictions)),
        "confusion_matrix": confusion_matrix(y.iloc[test_idx], predictions).tolist(),
        "train_membership_fingerprint": membership_fingerprint(train_idx),
        "test_membership_fingerprint": membership_fingerprint(test_idx),
    }

protocol_b_run_1 = evaluate_indices(
    df, canonical_groups, canonical_train_idx_1, canonical_test_idx_1,
    "B_deterministic_feature_group_aware_split_run_1",
)
protocol_b_run_2 = evaluate_indices(
    df, canonical_groups, canonical_train_idx_2, canonical_test_idx_2,
    "B_deterministic_feature_group_aware_split_run_2",
)

protocol_b_repeatability = {
    "train_membership_equal": bool(
        protocol_b_run_1["train_membership_fingerprint"] == protocol_b_run_2["train_membership_fingerprint"]
    ),
    "test_membership_equal": bool(
        protocol_b_run_1["test_membership_fingerprint"] == protocol_b_run_2["test_membership_fingerprint"]
    ),
    "metrics_equal": bool(all(
        protocol_b_run_1[k] == protocol_b_run_2[k]
        for k in ["accuracy", "f1", "precision", "recall", "confusion_matrix"]
    )),
    "shared_groups_zero_run_1": protocol_b_run_1["shared_feature_groups"] == 0,
    "shared_groups_zero_run_2": protocol_b_run_2["shared_feature_groups"] == 0,
}

protocol_b = dict(protocol_b_run_1)
protocol_b["protocol"] = "B_deterministic_feature_group_aware_split"
protocol_b["repeatability_check"] = protocol_b_repeatability
protocol_b["second_run"] = protocol_b_run_2
protocol_b


{'protocol': 'B_deterministic_feature_group_aware_split',
 'train_rows': 8797,
 'test_rows': 2258,
 'train_target_counts': {'0': 3966, '1': 4831},
 'test_target_counts': {'0': 932, '1': 1326},
 'shared_feature_groups': 0,
 'accuracy': 0.9481842338352524,
 'f1': 0.9554625047582794,
 'precision': 0.9646425826287471,
 'recall': 0.9464555052790347,
 'confusion_matrix': [[886, 46], [71, 1255]],
 'train_membership_fingerprint': '8ce409452de18f284e44c132f3628b4b929af2a5967b64d4da5ab1115a191760',
 'test_membership_fingerprint': '1189855d6296ad48d606e2fa10edc50f24d765ee77d3c38969f487d60b59bc35',
 'repeatability_check': {'train_membership_equal': True,
  'test_membership_equal': True,
  'metrics_equal': True,
  'shared_groups_zero_run_1': True,
  'shared_groups_zero_run_2': True},
 'second_run': {'protocol': 'B_deterministic_feature_group_aware_split_run_2',
  'train_rows': 8797,
  'test_rows': 2258,
  'train_target_counts': {'0': 3966, '1': 4831},
  'test_target_counts': {'0': 932, '1': 1326},



## 4. Protocol C — conflicting groups excluded

This is a **sensitivity analysis**, not a cleaning decision.

A conflicting feature group occurs when its 30-feature vector appears with both `Result=-1` and `Result=1`. Those groups are temporarily excluded to measure their influence on evaluation.

The canonical CSV remains unchanged.


In [9]:
grouped_labels = (
    df.groupby(feature_cols, dropna=False, sort=False)[TARGET]
      .nunique()
)
conflict_tuples = set(grouped_labels[grouped_labels > 1].index.tolist())

conflict_mask = tuple_series.isin(conflict_tuples)
df_consistent = df.loc[~conflict_mask].copy()

consistent_tuples = df_consistent[feature_cols].apply(tuple, axis=1)
consistent_unique = sorted(set(consistent_tuples.tolist()))
consistent_map = {t: i for i, t in enumerate(consistent_unique)}
consistent_groups = consistent_tuples.map(consistent_map).astype("int64")

splitter_c = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
train_idx_c, test_idx_c = next(
    splitter_c.split(
        df_consistent[feature_cols],
        df_consistent[TARGET],
        groups=consistent_groups,
    )
)

protocol_c = evaluate_indices(
    df_consistent,
    consistent_groups,
    train_idx_c,
    test_idx_c,
    "C_group_aware_conflicting_groups_excluded",
)

protocol_c.update({
    "rows_available": int(len(df_consistent)),
    "rows_excluded_from_canonical": int(len(df) - len(df_consistent)),
    "conflicting_groups": int(len(conflict_tuples)),
    "conflicting_row_rate": float(conflict_mask.mean()),
})
protocol_c


{'protocol': 'C_group_aware_conflicting_groups_excluded',
 'train_rows': 8586,
 'test_rows': 2112,
 'train_target_counts': {'0': 3795, '1': 4791},
 'test_target_counts': {'0': 958, '1': 1154},
 'shared_feature_groups': 0,
 'accuracy': 0.9621212121212122,
 'f1': 0.9652476107732406,
 'precision': 0.9677700348432056,
 'recall': 0.962738301559792,
 'confusion_matrix': [[921, 37], [43, 1111]],
 'train_membership_fingerprint': '2f3e46996ca7ada966528e624dc9395150c04a6252d52cb1643bf2828e1938b4',
 'test_membership_fingerprint': '5b157cb053a556f5bc70ef224df74a0fc8b7b8731161161837d47a919b64a9a8',
 'rows_available': 10698,
 'rows_excluded_from_canonical': 357,
 'conflicting_groups': 64,
 'conflicting_row_rate': 0.03229308005427409}


## 5. Protocol D — derived-label policy

A deterministic majority-label rule can be computed for a conflicting feature group, but it would replace observed labels with a label derived from the same dataset.

That is **not a neutral evaluation protocol** unless an external source or domain rule establishes that the majority label is authoritative.

Therefore Phase 3C records D as **inadmissible for the official benchmark** rather than manufacturing labels.


In [10]:

protocol_d = {
    "protocol": "D_deterministic_group_majority_label",
    "status": "not_admissible_as_official_benchmark",
    "reason": (
        "Would derive/rewrite labels for conflicting feature groups without "
        "external source or domain authority. This changes ground truth rather "
        "than merely changing the split."
    ),
    "production_dataset_modified": False,
}
protocol_d


{'protocol': 'D_deterministic_group_majority_label',
 'status': 'not_admissible_as_official_benchmark',
 'reason': 'Would derive/rewrite labels for conflicting feature groups without external source or domain authority. This changes ground truth rather than merely changing the split.',
 'production_dataset_modified': False}


## 6. Candidate protocol comparison

This table is descriptive. It is not a model ranking.

For protocol selection, the critical properties are:
- leakage control;
- reproducibility;
- preservation of observed labels;
- transparent conflict handling;
- stability of train/test membership.


In [11]:

comparison = pd.DataFrame([
    {
        "protocol": protocol_a["protocol"],
        "train_rows": protocol_a["train_rows"],
        "test_rows": protocol_a["test_rows"],
        "shared_groups": protocol_a["shared_feature_groups"],
        "duplicate_overlap_rate": protocol_a["test_duplicate_overlap_rate"],
        "f1": protocol_a["f1"],
        "precision": protocol_a["precision"],
        "recall": protocol_a["recall"],
    },
    {
        "protocol": protocol_b["protocol"],
        "train_rows": protocol_b["train_rows"],
        "test_rows": protocol_b["test_rows"],
        "shared_groups": protocol_b["shared_feature_groups"],
        "duplicate_overlap_rate": 0.0,
        "f1": protocol_b["f1"],
        "precision": protocol_b["precision"],
        "recall": protocol_b["recall"],
    },
    {
        "protocol": protocol_c["protocol"],
        "train_rows": protocol_c["train_rows"],
        "test_rows": protocol_c["test_rows"],
        "shared_groups": protocol_c["shared_feature_groups"],
        "duplicate_overlap_rate": 0.0,
        "f1": protocol_c["f1"],
        "precision": protocol_c["precision"],
        "recall": protocol_c["recall"],
    },
])
comparison


,protocol,train_rows,test_rows,shared_groups,duplicate_overlap_rate,f1,precision,recall
0,A_random_row_split,8844,2211,1143,0.654455,0.972332,0.964706,0.980080
1,B_deterministic_feature_group_aware_split,8797,2258,0,0.000000,0.955463,0.964643,0.946456
2,C_group_aware_conflicting_groups_excluded,8586,2112,0,0.000000,0.965248,0.967770,0.962738


## 7. Freeze decision

Protocol B is frozen as the primary benchmark only if all of the following are true:

1. canonical group construction is valid and contiguous;
2. repeated Protocol B runs have identical train/test membership fingerprints;
3. repeated Protocol B runs have identical metrics;
4. both runs have zero shared feature groups.

Protocol A remains the historical random-split baseline. Protocol C remains sensitivity analysis. Protocol D remains inadmissible without external label authority.


In [12]:
primary_freeze_checks = {
    "canonical_group_relation_valid": group_equivalence_check["canonical_valid"],
    "canonical_group_ids_contiguous": group_equivalence_check["canonical_group_ids_are_contiguous"],
    "repeat_train_membership_equal": protocol_b_repeatability["train_membership_equal"],
    "repeat_test_membership_equal": protocol_b_repeatability["test_membership_equal"],
    "repeat_metrics_equal": protocol_b_repeatability["metrics_equal"],
    "run_1_shared_groups_zero": protocol_b_repeatability["shared_groups_zero_run_1"],
    "run_2_shared_groups_zero": protocol_b_repeatability["shared_groups_zero_run_2"],
}

deterministic_freeze_passed = all(primary_freeze_checks.values())

decision = {
    "official_primary_protocol": (
        "B_deterministic_feature_group_aware_split"
        if deterministic_freeze_passed
        else "UNRESOLVED"
    ),
    "historical_baseline": "A_random_row_split",
    "sensitivity_protocol": "C_group_aware_conflicting_groups_excluded",
    "derived_label_protocol": "D_not_admissible_without_external_label_authority",
    "deterministic_split_reproducible": bool(deterministic_freeze_passed),
    "freeze_checks": primary_freeze_checks,
    "production_code_modified": False,
    "canonical_dataset_modified": False,
}
decision


{'official_primary_protocol': 'B_deterministic_feature_group_aware_split',
 'historical_baseline': 'A_random_row_split',
 'sensitivity_protocol': 'C_group_aware_conflicting_groups_excluded',
 'derived_label_protocol': 'D_not_admissible_without_external_label_authority',
 'deterministic_split_reproducible': True,
 'freeze_checks': {'canonical_group_relation_valid': True,
  'canonical_group_ids_contiguous': True,
  'repeat_train_membership_equal': True,
  'repeat_test_membership_equal': True,
  'repeat_metrics_equal': True,
  'run_1_shared_groups_zero': True,
  'run_2_shared_groups_zero': True},
 'production_code_modified': False,
 'canonical_dataset_modified': False}

In [13]:
artifact = {
    "experiment": {
        "name": "phase_3c1_evaluation_protocol_freeze",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "completed" if deterministic_freeze_passed else "failed_freeze_checks",
        "production_code_modified": False,
        "canonical_dataset_modified": False,
        "notebook": str(NOTEBOOK_PATH),
    },
    "dataset": {
        "path": str(DATA_PATH),
        "sha256": actual_sha256,
        "shape": list(df.shape),
        "dataframe_fingerprint": dataframe_fingerprint(df),
        "target": TARGET,
        "feature_count": len(feature_cols),
        "rows_participating_in_duplicate_feature_groups": int(df.duplicated(subset=feature_cols, keep=False).sum()),
        "unique_full_rows": int(len(df.drop_duplicates())),
    },
    "configuration": {
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "group_encoding": "lexicographically_sorted_complete_feature_tuple_to_contiguous_integer_id",
        "target_mapping": {"-1": 0, "1": 1},
        "preprocessing": {"transformer": "KNNImputer", "n_neighbors": 3, "weights": "uniform"},
        "model": {
            "type": "RandomForestClassifier",
            "n_estimators": 128,
            "criterion": "gini",
            "bootstrap": True,
            "max_depth": None,
            "max_features": "sqrt",
            "random_state": RANDOM_STATE,
        },
    },
    "reconciliation": reconciliation,
    "group_equivalence": group_equivalence_check,
    "protocols": {
        "A_random_row_split": protocol_a,
        "B_deterministic_feature_group_aware_split": protocol_b,
        "C_conflicting_groups_excluded_sensitivity": protocol_c,
        "D_derived_label_policy": protocol_d,
    },
    "decision": decision,
    "notes": [
        "Protocol A is retained as a historical random-row baseline and is not leakage-controlled for duplicate feature groups.",
        "Different group-ID orderings can produce different seeded GroupShuffleSplit partitions even when they encode the same mathematical groups.",
        "Protocol B uses one canonical group-ID construction and freezes only after repeated membership and metric checks pass.",
        "Protocol C excludes conflicting feature groups only for sensitivity analysis; the canonical dataset is unchanged.",
        "Protocol D is not an official benchmark because it derives/reassigns labels without external authority.",
    ],
}

ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATH.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
print("Wrote:", ARTIFACT_PATH)
print(json.dumps(decision, indent=2))


Wrote: E:\Projects\Network security log triage agent\notebooks\evaluation\phase_3c_evaluation_protocol_comparison_freezed.json
{
  "official_primary_protocol": "B_deterministic_feature_group_aware_split",
  "historical_baseline": "A_random_row_split",
  "sensitivity_protocol": "C_group_aware_conflicting_groups_excluded",
  "derived_label_protocol": "D_not_admissible_without_external_label_authority",
  "deterministic_split_reproducible": true,
  "freeze_checks": {
    "canonical_group_relation_valid": true,
    "canonical_group_ids_contiguous": true,
    "repeat_train_membership_equal": true,
    "repeat_test_membership_equal": true,
    "repeat_metrics_equal": true,
    "run_1_shared_groups_zero": true,
    "run_2_shared_groups_zero": true
  },
  "production_code_modified": false,
  "canonical_dataset_modified": false
}
